# 02 - Classical Baseline Model

## RBF Kernel SVM with GridSearchCV

This notebook trains a classical Support Vector Machine with RBF kernel as a baseline for comparison with the quantum kernel SVM.

## 1. Classical Kernel Methods

### What are Kernel Methods?

Kernel methods are a class of algorithms for pattern analysis that operate in an implicit, high-dimensional feature space. The key idea is to compute similarity between data points in a way that doesn't require explicitly computing the transformation.

### The Kernel Trick

Instead of mapping data to a high-dimensional space explicitly, kernel methods use a **kernel function** k(x, z) that computes the inner product in the feature space:

$$k(x, z) = \langle \phi(x), \phi(z) \rangle$$

### Popular Kernels

1. **Linear Kernel**: k(x, z) = x · z
2. **Polynomial Kernel**: k(x, z) = (γx·z + r)^d
3. **RBF (Gaussian) Kernel**: k(x, z) = exp(-γ||x - z||²)

### Why RBF Kernel?

The RBF kernel is one of the most popular kernels because:
- It can map data to infinite-dimensional space
- It has fewer hyperparameters than polynomial kernel
- It handles non-linear relationships well

## 2. Setup and Load Preprocessed Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
import sys
import os

# Add src to path
sys.path.append(os.path.join(os.path.dirname('../'), 'src'))

# Set random seed
np.random.seed(42)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

In [ ]:
# Load preprocessed data
X_train_pca = np.load('../results/X_train_pca.npy')
X_test_pca = np.load('../results/X_test_pca.npy')
y_train = np.load('../results/y_train.npy')
y_test = np.load('../results/y_test.npy')

print(f"Training data: {X_train_pca.shape}")
print(f"Test data: {X_test_pca.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test labels: {y_test.shape}")

## 3. Train Classical RBF SVM

### 3.1 Understanding SVM Hyperparameters

**C (Regularization Parameter)**
- Controls the trade-off between maximizing the margin and minimizing the classification error
- Small C: Large margin, more misclassifications
- Large C: Small margin, fewer misclassifications

**Gamma (Kernel Coefficient)**
- Defines how far the influence of a single training example reaches
- Small gamma: Far reach (consider many nearby points)
- Large gamma: Short reach (only nearby points matter)
- 'scale': 1 / (n_features * X.var())
- 'auto': 1 / n_features

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

# Define parameter grid for GridSearchCV
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01]
}

print("Parameter Grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

In [ ]:
# Create SVM classifier with RBF kernel
svm_rbf = SVC(kernel='rbf', random_state=42)

# Perform GridSearchCV
print("\nPerforming GridSearchCV with 3-fold cross-validation...")
start_time = time.time()

grid_search = GridSearchCV(
    estimator=svm_rbf,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_pca, y_train)

grid_search_time = time.time() - start_time
print(f"\nGridSearchCV completed in {grid_search_time:.2f} seconds")

In [ ]:
# Display best parameters
print("\nBest Parameters:")
print(grid_search.best_params_)

print(f"\nBest CV Score: {grid_search.best_score_:.4f}")

### 3.2 Train with Best Parameters

In [ ]:
# Get the best model
best_svm = grid_search.best_estimator_

# Train on full training set
start_time = time.time()
best_svm.fit(X_train_pca, y_train)
train_time = time.time() - start_time

print(f"Training completed in {train_time:.4f} seconds")
print(f"Number of support vectors: {best_svm.n_support_}")
print(f"Total support vectors: {sum(best_svm.n_support_)}")

## 4. Model Evaluation

In [ ]:
# Make predictions
y_pred_train = best_svm.predict(X_train_pca)
y_pred_test = best_svm.predict(X_test_pca)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)

print("=" * 60)
print("CLASSICAL SVM RESULTS")
print("=" * 60)
print(f"\nTraining Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Detailed metrics on test set
precision = precision_score(y_test, y_pred_test, pos_label=4)
recall = recall_score(y_test, y_pred_test, pos_label=4)
f1 = f1_score(y_test, y_pred_test, pos_label=4)

print("\nTest Set Metrics (Digit 4 as positive class):")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1 Score: {f1:.4f}")

In [ ]:
# Full classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_test, target_names=['Digit 4', 'Digit 9']))

## 5. Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Digit 4', 'Digit 9'],
            yticklabels=['Digit 4', 'Digit 9'],
            ax=ax)

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix - Classical RBF SVM', fontsize=14)

plt.tight_layout()
plt.savefig('../results/confusion_matrix_classical.png', dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to results/confusion_matrix_classical.png")

## 6. GridSearch Results Visualization

In [ ]:
# Visualize GridSearchCV results
cv_results = pd.DataFrame(grid_search.cv_results_)

# Get unique C and gamma values
C_values = param_grid['C']
gamma_values = param_grid['gamma']

# Create heatmap of mean test scores
score_matrix = cv_results.pivot(index='param_C', columns='param_gamma', values='mean_test_score')

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(score_matrix, annot=True, fmt='.3f', cmap='viridis', ax=ax)
ax.set_title('GridSearchCV Results - Mean CV Accuracy', fontsize=14)
ax.set_xlabel('Gamma')
ax.set_ylabel('C')

plt.tight_layout()
plt.savefig('../results/gridsearch_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("GridSearchCV heatmap saved to results/gridsearch_heatmap.png")

## 7. Save Results

In [ ]:
# Compile results
classical_results = {
    'model': 'Classical RBF SVM',
    'best_params': grid_search.best_params_,
    'best_cv_score': grid_search.best_score_,
    'train_accuracy': train_accuracy,
    'test_accuracy': test_accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'train_time': train_time,
    'grid_search_time': grid_search_time,
    'n_support_vectors': int(sum(best_svm.n_support_)),
    'confusion_matrix': cm.tolist()
}

# Save results
import json

with open('../results/classical_svm_results.json', 'w') as f:
    json.dump(classical_results, f, indent=2)

print("Classical SVM results saved to results/classical_svm_results.json")

## 8. Summary

In [ ]:
print("=" * 60)
print("CLASSICAL BASELINE MODEL - SUMMARY")
print("=" * 60)
print(f"\nBest Hyperparameters:")
print(f"  C: {grid_search.best_params_['C']}")
print(f"  Gamma: {grid_search.best_params_['gamma']}")
print(f"\nPerformance Metrics:")
print(f"  Training Accuracy: {train_accuracy:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall: {recall:.4f}")
print(f"  F1 Score: {f1:.4f}")
print(f"\nTiming:")
print(f"  GridSearchCV Time: {grid_search_time:.2f}s")
print(f"  Training Time: {train_time:.4f}s")
print("=" * 60)

---

## Next Steps

Proceed to **03_quantum_kernel_svm.ipynb** to train and evaluate the quantum kernel SVM model.